In [9]:
'''
Program: Toy spelling checker application using the bottom-up Levenshtein Edit Distance (LED) algorithm:
Author: Yaseer Sabir
Date: 11/14/2025

'''

# sys is used for command-line argument handling
# re isregular expressions used to extract words from text
# Counter efficiently counts word frequencies to build dictionary
import sys
import re
from collections import Counter

# When running inside Jupyter Notebook, Python is launched with hidden system arguments such as "-f <kernel.json>". These are NOT meant to
# be read by our program, and they would cause int() conversion to fail.
#
# Therefore: If we detect any argument starting with "-f", we assume we are in Jupyter and set a safe default value for MAX_SUG. Otherwise, we are likely running from the command line and can read
# the user-provided integer (e.g., python main.py 7).
# This makes the program work correctly in BOTH environments.
if any(arg.startswith("-f") for arg in sys.argv):
    MAX_SUG = 7   # Safe default when running inside a Jupyter Notebook
else:
    if len(sys.argv) >= 2:       # User supplied an argument in the terminal
        try:
            MAX_SUG = int(sys.argv[1])
        except ValueError:
            MAX_SUG = 7          # Fallback if argument is invalid
    else:
        MAX_SUG = 7              # Default if no argument is provided

MIN_LEN = 3      # minimum length of suggested word
MAX_ED  = 2      # maximum edit distance allowed

# re.findall(...) extracts all alphabetic words from the file
# .lower() ensures all words are case-insensitive
# Counter(...) automatically counts frequency of each word
# The result is a dictionary-like object: {word: frequency}
dictionary = Counter(re.findall(r"[a-zA-Z]+", open("dictionary.txt").read().lower()))
print(f"Dictionary size: {len(dictionary)}")

# Again we use regex to extract words, ignoring punctuation.
# Words are converted to lowercase for consistent matching.
misspelled_words = re.findall(r"[a-zA-Z]+", open("misspelled.txt").read().lower())
print("\nChecking file: misspelled.txt\n")

# This function computes LED between strings a and b using dynamic programming.
# LED counts the minimum number of operations needed to transform a → b.
# Operations allowed: Insertion, Deletion and Substitution
# dp[i][j] = minimum edit distance between:
#   a[:i] (first i characters of a)
#   b[:j] (first j characters of b)
def led(a, b):
    m, n = len(a), len(b)

    # Create an (m+1)×(n+1) matrix initialized to zeros
    dp = [[0] * (n + 1) for _ in range(m + 1)]

    # Base cases:
    # Distance from a prefix to empty string = number of deletions
    for i in range(m + 1):
        dp[i][0] = i

    # Distance from empty string to b prefix = number of insertions
    for j in range(n + 1):
        dp[0][j] = j

    # Fill the DP table bottom-up
    for i in range(1, m + 1):
        for j in range(1, n + 1):

            # No cost if characters match, else cost = 1
            cost = 0 if a[i - 1] == b[j - 1] else 1

            # dp recurrence:
            dp[i][j] = min(
                dp[i - 1][j] + 1,       # deletion
                dp[i][j - 1] + 1,       # insertion
                dp[i - 1][j - 1] + cost # substitution
            )

    # Final answer: distance between full strings a and b
    return dp[m][n]

# This function receives a misspelled word and searches the dictionary
# for possible corrections. A word is considered a valid suggestion if:
#   1. Its length is >= MIN_LEN
#   2. Its Levenshtein Edit Distance (LED) from the misspelled word
#      is <= MAX_ED
# Each suggestion is stored as a tuple:
#       (word, (edit_distance, frequency))
# After collecting all candidates, the function sorts them by:
#   - Primary key: edit distance (ascending)
#   - Secondary key: frequency (descending)
# Finally, it returns up to MAX_SUG suggestions.
def get_suggestions(word):
    suggestions = []

    # Loop through every dictionary word and its frequency
    for w, freq in dictionary.items():

        # Skip very short words (constraint #1)
        if len(w) < MIN_LEN:
            continue

        # Compute edit distance between misspelled word and candidate
        d = led(word, w)

        # Keep word only if it meets max edit distance constraint
        if d <= MAX_ED:
            suggestions.append((w, (d, freq)))

    # Sort by (edit distance ASC, frequency DESC)
    # x[1][0] = edit distance
    # x[1][1] = frequency
    suggestions.sort(key=lambda x: (x[1][0], -x[1][1]))

    # Return only the top MAX_SUG best matches
    return suggestions[:MAX_SUG]


# This loop processes each word from misspelled.txt.
# If a word is NOT found in the dictionary, we:
#   1. Generate suggestions using get_suggestions()
#   2. Format them into a dictionary for clean printing
#   3. Display them in the required output format
# ---------------------------------------------------------------
for word in misspelled_words:

    # If the word is spelled correctly (exists in dictionary), skip it
    if word in dictionary:
        continue

    # Get a sorted list of possible corrections
    suggestions = get_suggestions(word)

    # Convert list of tuples into a nicer dictionary for printing
    formatted = {w: data for w, data in suggestions}

    # Print the final suggestions for this misspelled word
    print(f"- Suggestions for '{word}': {formatted}\n")


Dictionary size: 29157

Checking file: misspelled.txt

- Suggestions for 'homan': {'woman': (1, 325), 'human': (1, 170), 'coman': (1, 17), 'roman': (1, 8), 'man': (2, 1652), 'women': (2, 390), 'home': (2, 295)}

- Suggestions for 'spote': {'spoke': (1, 218), 'spite': (1, 117), 'spot': (1, 76), 'spots': (1, 12), 'smote': (1, 4), 'spore': (1, 3), 'some': (2, 1536)}

- Suggestions for 'belst': {'best': (1, 268), 'beast': (1, 26), 'belt': (1, 12), 'felt': (2, 697), 'west': (2, 286), 'rest': (2, 209), 'else': (2, 201)}

- Suggestions for 'effrts': {'efforts': (1, 103), 'effort': (2, 130), 'effects': (2, 82), 'forts': (2, 8), 'exerts': (2, 3), 'effete': (2, 1)}

- Suggestions for 'speling': {'spelling': (1, 4), 'feeling': (2, 362), 'seeing': (2, 207), 'speaking': (2, 185), 'swelling': (2, 167), 'smiling': (2, 161), 'opening': (2, 146)}

- Suggestions for 'perrfect': {'perfect': (1, 39), 'prefect': (2, 2)}

- Suggestions for 'avorage': {'average': (1, 18), 'voyage': (2, 12), 'forage': (2, 7),

In [11]:
'''
Program: Minimal Changes Program
Author: Yaseer Sabir
'''
def make_change(amount):
    """
    Compute the least number of bills/coins needed to make change
    for a given amount of money using the greedy algorithm.

    The function converts dollars to cents to avoid floating-point
    precision issues, then repeatedly selects the largest denomination
    that does not exceed the remaining amount.
    """
    
    # Convert dollars to cents (avoids floating point rounding errors)
    cents = round(amount * 100)

    # List of denominations from largest → smallest
    # Each item is a tuple: (label, value in cents)
    denominations = [
        ("$100", 10000),
        ("$50",   5000),
        ("$20",   2000),
        ("$10",   1000),
        ("$5",     500),
        ("$1",     100),
        ("50¢",     50),
        ("25¢",     25),
        ("10¢",     10),
        ("5¢",       5),
        ("1¢",       1)
    ]

    result = {}  # store the denomination → count

    # Greedy algorithm: choose the largest possible denomination first
    for name, value in denominations:
        count = cents // value  # number of this denomination needed
        if count > 0:
            result[name] = count
            cents %= value     # reduce remaining amount

    return result


# ---------------------------------------------------------------
# MAIN PROGRAM EXECUTION
# ---------------------------------------------------------------
# Prompt user to enter an amount
amount = float(input("Enter an amount: $"))

# Compute the optimal set of bills/coins
change = make_change(amount)

# Display the result in a clean format
print("\nChange breakdown:")
for denom, count in change.items():
    print(f"{denom}: {count}")

Enter an amount: $ 269.63



Change breakdown:
$100: 2
$50: 1
$10: 1
$5: 1
$1: 4
50¢: 1
10¢: 1
1¢: 3
